In [ ]:
# =========================
# DRIVE WARM-UP — force sync before any path operations
# =========================
import subprocess, time

def force_drive_sync(*paths):
    """Touch every folder so Drive fetches its metadata."""
    for p in paths:
        p = Path(p)
        if p.exists():
            try:
                # listing forces metadata fetch
                children = list(p.iterdir())
                print(f"  synced: {p}  ({len(children)} entries visible)")
            except Exception as e:
                print(f"  WARNING: could not list {p}: {e}")
        else:
            print(f"  MISSING: {p}")

print("Warming up Drive mount — this can take 30-120 s on large datasets...")
warm_targets = [
    TN5000_ROOT, TN5000_ANN, TN5000_IMGSETS_MAIN, TN5000_JPEG,
    BATCH1_ROOT, BATCH1_DATASET,
    BATCH2_ROOT, BATCH2_DATASET,
]
force_drive_sync(*warm_targets)

# Give Drive a moment to finish streaming metadata
time.sleep(5)
print("Warm-up done.")

Warming up Drive mount — this can take 30-120 s on large datasets...
  MISSING: /content/drive/MyDrive/1/TN5000_forReview
  MISSING: /content/drive/MyDrive/1/TN5000_forReview/Annotations
  MISSING: /content/drive/MyDrive/1/TN5000_forReview/ImageSets/Main
  MISSING: /content/drive/MyDrive/1/TN5000_forReview/JPEGImages
  MISSING: /content/drive/MyDrive/2/batch1_image
  MISSING: /content/drive/MyDrive/2/batch1_image/dataset
  MISSING: /content/drive/MyDrive/2/batch2_image
  MISSING: /content/drive/MyDrive/2/batch2_image/dataset
Warm-up done.


In [ ]:
# =========================
# CELL 1 — Environment + dataset path audit
# =========================

from google.colab import drive
drive.mount('/content/drive',force_remount= True)

import os, sys, glob, json, random, platform
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
    TORCH_AVAILABLE = True
except Exception as e:
    TORCH_AVAILABLE = False
    print("Torch import failed:", e)

# -------------------------
# Fixed project paths
# -------------------------
DRIVE_ROOT = Path("/content/drive/MyDrive")

TN5000_ROOT = DRIVE_ROOT / "1" / "TN5000_forReview"
TN5000_ANN = TN5000_ROOT / "Annotations"
TN5000_IMGSETS_MAIN = TN5000_ROOT / "ImageSets" / "Main"
TN5000_JPEG = TN5000_ROOT / "JPEGImages"

EXT_ROOT = DRIVE_ROOT / "2"

BATCH1_ROOT = EXT_ROOT / "batch1_image"
BATCH1_DATASET = BATCH1_ROOT / "dataset"
BATCH1_CSV = BATCH1_ROOT / "batch1_image.csv"
BATCH1_LABEL_CSV = BATCH1_ROOT / "batch1_image_label.csv"

BATCH2_ROOT = EXT_ROOT / "batch2_image"
BATCH2_DATASET = BATCH2_ROOT / "dataset"
BATCH2_CSV = BATCH2_ROOT / "batch2_image.csv"
BATCH2_LABEL_CSV = BATCH2_ROOT / "batch2_image_label.csv"

# Output directory for manuscript-grade artifacts later

OUT_ROOT = DRIVE_ROOT / "thyroid_external_validation_outputs"

# Use os.makedirs with a retry — Drive mount can be briefly unresponsive
for attempt in range(3):
    try:
        os.makedirs(OUT_ROOT, exist_ok=True)
        # Verify it actually exists
        assert OUT_ROOT.exists(), "mkdir reported success but path still missing"
        print("OUT_ROOT ready:", OUT_ROOT)
        break
    except Exception as e:
        print(f"Attempt {attempt+1} failed: {e}. Retrying in 5s...")
        time.sleep(5)
else:
    raise RuntimeError(f"Could not create OUT_ROOT after 3 attempts: {OUT_ROOT}")

FIG_DPI = 300
RANDOM_SEEDS = [42, 123, 2024, 3407, 777]

# -------------------------
# Helper functions
# -------------------------
def path_status(p: Path):
    return {
        "path": str(p),
        "exists": p.exists(),
        "is_dir": p.is_dir(),
        "is_file": p.is_file()
    }

def count_files(folder: Path, patterns):
    counts = {}
    total = 0
    for pat in patterns:
        n = len(list(folder.rglob(pat))) if folder.exists() else 0
        counts[pat] = n
        total += n
    counts["TOTAL"] = total
    return counts

def preview_text_file(p: Path, n=8):
    print(f"\n--- Preview: {p} ---")
    if not p.exists():
        print("MISSING")
        return
    try:
        with open(p, "r", encoding="utf-8", errors="ignore") as f:
            for i, line in enumerate(f):
                if i >= n:
                    break
                print(line.rstrip())
    except Exception as e:
        print("Could not preview:", repr(e))

def read_csv_preview(p: Path, name: str):
    print(f"\n==================== {name} ====================")
    print("Path:", p)
    if not p.exists():
        print("MISSING")
        return None
    try:
        df = pd.read_csv(p)
        print("Shape:", df.shape)
        print("Columns:", list(df.columns))
        display(df.head(10))
        return df
    except Exception as e:
        print("CSV read failed:", repr(e))
        return None

# -------------------------
# Environment report
# -------------------------
print("===== ENVIRONMENT =====")
print("Python:", sys.version)
print("Platform:", platform.platform())

if TORCH_AVAILABLE:
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA version:", torch.version.cuda)
else:
    print("PyTorch not available.")

print("Output root:", OUT_ROOT)
print("Figure DPI locked at:", FIG_DPI)
print("Planned seeds:", RANDOM_SEEDS)

# -------------------------
# Path existence audit
# -------------------------
print("\n===== PATH AUDIT =====")
all_paths = [
    TN5000_ROOT, TN5000_ANN, TN5000_IMGSETS_MAIN, TN5000_JPEG,
    BATCH1_ROOT, BATCH1_DATASET, BATCH1_CSV, BATCH1_LABEL_CSV,
    BATCH2_ROOT, BATCH2_DATASET, BATCH2_CSV, BATCH2_LABEL_CSV
]

for p in all_paths:
    print(json.dumps(path_status(p), indent=2))

# -------------------------
# File count audit
# -------------------------
print("\n===== FILE COUNTS =====")
image_patterns = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff", "*.JPG", "*.JPEG", "*.PNG"]

print("\nTN5000 JPEGImages counts:")
print(count_files(TN5000_JPEG, image_patterns))

print("\nTN5000 Annotations XML counts:")
print(count_files(TN5000_ANN, ["*.xml", "*.XML"]))

print("\nExternal Batch 1 image counts:")
print(count_files(BATCH1_DATASET, image_patterns))

print("\nExternal Batch 2 image counts:")
print(count_files(BATCH2_DATASET, image_patterns))

# -------------------------
# TN5000 ImageSets/Main audit
# -------------------------
print("\n===== TN5000 ImageSets/Main FILES =====")
if TN5000_IMGSETS_MAIN.exists():
    imgset_files = sorted([p for p in TN5000_IMGSETS_MAIN.iterdir() if p.is_file()])
    for p in imgset_files:
        try:
            n_lines = sum(1 for _ in open(p, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            n_lines = "unreadable"
        print(f"{p.name} | lines: {n_lines}")
    for p in imgset_files[:10]:
        preview_text_file(p, n=8)
else:
    print("TN5000 ImageSets/Main folder missing.")

# -------------------------
# External CSV audit
# -------------------------
b1_img_df = read_csv_preview(BATCH1_CSV, "Batch 1 image CSV")
b1_lbl_df = read_csv_preview(BATCH1_LABEL_CSV, "Batch 1 label CSV")
b2_img_df = read_csv_preview(BATCH2_CSV, "Batch 2 image CSV")
b2_lbl_df = read_csv_preview(BATCH2_LABEL_CSV, "Batch 2 label CSV")

# -------------------------
# Save initial config for reproducibility
# -------------------------
config = {
    "TN5000_ROOT": str(TN5000_ROOT),
    "TN5000_ANN": str(TN5000_ANN),
    "TN5000_IMGSETS_MAIN": str(TN5000_IMGSETS_MAIN),
    "TN5000_JPEG": str(TN5000_JPEG),
    "BATCH1_ROOT": str(BATCH1_ROOT),
    "BATCH1_DATASET": str(BATCH1_DATASET),
    "BATCH1_CSV": str(BATCH1_CSV),
    "BATCH1_LABEL_CSV": str(BATCH1_LABEL_CSV),
    "BATCH2_ROOT": str(BATCH2_ROOT),
    "BATCH2_DATASET": str(BATCH2_DATASET),
    "BATCH2_CSV": str(BATCH2_CSV),
    "BATCH2_LABEL_CSV": str(BATCH2_LABEL_CSV),
    "OUT_ROOT": str(OUT_ROOT),
    "FIG_DPI": FIG_DPI,
    "RANDOM_SEEDS": RANDOM_SEEDS
}

with open(OUT_ROOT / "cell1_config_paths.json", "w") as f:
    json.dump(config, f, indent=2)

print("\nSaved config to:", OUT_ROOT / "cell1_config_paths.json")
print("\nCELL 1 COMPLETE. Paste the full output here before running the next cell.")

Mounted at /content/drive
OUT_ROOT ready: /content/drive/MyDrive/thyroid_external_validation_outputs
===== ENVIRONMENT =====
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
Output root: /content/drive/MyDrive/thyroid_external_validation_outputs
Figure DPI locked at: 300
Planned seeds: [42, 123, 2024, 3407, 777]

===== PATH AUDIT =====
{
  "path": "/content/drive/MyDrive/1/TN5000_forReview",
  "exists": true,
  "is_dir": true,
  "is_file": false
}
{
  "path": "/content/drive/MyDrive/1/TN5000_forReview/Annotations",
  "exists": true,
  "is_dir": true,
  "is_file": false
}
{
  "path": "/content/drive/MyDrive/1/TN5000_forReview/ImageSets/Main",
  "exists": true,
  "is_dir": true,
  "is_file": false
}
{
  "path": "/content/drive/MyDrive/1/TN5000_forReview/JPEGImages",
  "exists": true,
  "is_dir": true,
  "is_file": false
}
{
  "path": "/content/drive/MyDrive

,patient_name,path
0,341,341_001.Jpg
1,341,341_002.Jpg
2,341,341_004.Jpg
3,341,341_005.Jpg
4,341,341_006.Jpg
5,341,341_007.Jpg
6,341,341_008.Jpg
7,341,341_010.Jpg
8,239,239_001.Jpg
9,239,239_002.Jpg



==================== Batch 1 label CSV ====================
Path: /content/drive/MyDrive/2/batch1_image/batch1_image_label.csv
Shape: (601, 2)
Columns: ['patient_name', 'histo_label']


,patient_name,histo_label
0,502,0
1,288,1
2,323,0
3,178,0
4,427,1
5,374,1
6,553,0
7,129,0
8,506,1
9,419,1



==================== Batch 2 image CSV ====================
Path: /content/drive/MyDrive/2/batch2_image/batch2_image.csv
Shape: (2503, 2)
Columns: ['path', 'patient_name']


,path,patient_name
0,195_001_173546.Jpg,195
1,195_002_173547.Jpg,195
2,195_003_173547.Jpg,195
3,195_004_173547.Jpg,195
4,195_005_173547.Jpg,195
5,195_006_173547.Jpg,195
6,195_007_173548.Jpg,195
7,40_001_173733.Jpg,40
8,40_002_173733.Jpg,40
9,40_003_173733.Jpg,40



==================== Batch 2 label CSV ====================
Path: /content/drive/MyDrive/2/batch2_image/batch2_image_label.csv
Shape: (241, 2)
Columns: ['patient_name', 'histo_label']


,patient_name,histo_label
0,0,0
1,1,0
2,2,0
3,3,1
4,4,1
5,5,1
6,6,0
7,7,1
8,8,0
9,9,1



Saved config to: /content/drive/MyDrive/thyroid_external_validation_outputs/cell1_config_paths.json

CELL 1 COMPLETE. Paste the full output here before running the next cell.


In [ ]:
# =========================
# CELL 2 — Build locked dataset manifests + label/split audit
# =========================

import os, re, json, hashlib, xml.etree.ElementTree as ET, shutil, time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image

# -------------------------
# STEP 1: Declare all Drive paths
# -------------------------
DRIVE_ROOT = Path("/content/drive/MyDrive")

TN5000_ROOT         = DRIVE_ROOT / "1" / "TN5000_forReview"
TN5000_ANN          = TN5000_ROOT / "Annotations"
TN5000_IMGSETS_MAIN = TN5000_ROOT / "ImageSets" / "Main"
TN5000_JPEG         = TN5000_ROOT / "JPEGImages"

EXT_ROOT        = DRIVE_ROOT / "2"
BATCH1_ROOT     = EXT_ROOT / "batch1_image"
BATCH1_DATASET  = BATCH1_ROOT / "dataset"
BATCH1_CSV      = BATCH1_ROOT / "batch1_image.csv"
BATCH1_LABEL_CSV= BATCH1_ROOT / "batch1_image_label.csv"
BATCH2_ROOT     = EXT_ROOT / "batch2_image"
BATCH2_DATASET  = BATCH2_ROOT / "dataset"
BATCH2_CSV      = BATCH2_ROOT / "batch2_image.csv"
BATCH2_LABEL_CSV= BATCH2_ROOT / "batch2_image_label.csv"

OUT_ROOT = DRIVE_ROOT / "thyroid_external_validation_outputs"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# -------------------------
# STEP 2: Copy everything to local /tmp (fast local disk)
# -------------------------
def sync_to_local(src, dst, suffix, desc):
    src, dst = Path(src), Path(dst)
    files = list(src.rglob(f"*{suffix}")) if src.exists() else []
    cached = list(dst.glob(f"*{suffix}")) if dst.exists() else []
    if len(cached) >= len(files) > 0:
        print(f"  {desc}: already cached ({len(cached)} files)")
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    print(f"  Copying {len(files)} {desc} files to local disk...")
    t0 = time.time()
    for p in files:
        shutil.copy2(p, dst / p.name)
    print(f"  Done in {time.time()-t0:.1f}s")
    return dst

print("=== CACHING TO LOCAL DISK (one-time, ~2-5 min) ===")
LOCAL_ANN  = sync_to_local(TN5000_ANN,     "/tmp/tn5000_ann",  ".xml", "TN5000 XMLs")
LOCAL_JPEG = sync_to_local(TN5000_JPEG,    "/tmp/tn5000_jpeg", ".jpg", "TN5000 JPEGs")
LOCAL_B1   = sync_to_local(BATCH1_DATASET, "/tmp/batch1",      ".Jpg", "Batch1 images")
LOCAL_B2   = sync_to_local(BATCH2_DATASET, "/tmp/batch2",      ".Jpg", "Batch2 images")
print("All cached.\n")

# -------------------------
# STEP 3: Helpers
# -------------------------
VALID_IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def list_images_case_insensitive(root: Path):
    if not root.exists():
        return []
    return sorted([p for p in root.rglob("*")
                   if p.is_file() and p.suffix.lower() in VALID_IMAGE_SUFFIXES])

def read_split_ids(split_file: Path):
    ids = []
    with open(split_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if s:
                ids.append(s)
    return ids

def normalize_label_name(x):
    return str(x).strip().lower().replace(" ", "").replace("_", "").replace("-", "")

def infer_binary_label_from_xml(xml_path: Path):
    root = ET.parse(xml_path).getroot()
    filename_node = root.find("filename")
    filename_xml = filename_node.text.strip() if filename_node is not None and filename_node.text else None

    object_names, bboxes = [], []
    for obj in root.findall(".//object"):
        name_node = obj.find("name")
        if name_node is not None and name_node.text is not None:
            object_names.append(name_node.text.strip())
        bb = obj.find("bndbox")
        if bb is not None:
            coords = []
            for tag in ["xmin", "ymin", "xmax", "ymax"]:
                node = bb.find(tag)
                try:
                    coords.append(float(node.text))
                except Exception:
                    coords.append(np.nan)
            bboxes.append(coords)

    norm_names = [normalize_label_name(n) for n in object_names]
    malignant_terms = {"malignant","malignancy","cancer","carcinoma",
                       "papillarycarcinoma","ptc","1","positive"}
    benign_terms    = {"benign","adenoma","nodulargoiter","goiter",
                       "thyroiditis","0","negative"}

    has_malignant = any(n in malignant_terms or "malignant" in n or "carcinoma" in n
                        for n in norm_names)
    has_benign    = any(n in benign_terms or "benign" in n for n in norm_names)

    if has_malignant and not has_benign:
        label = 1
    elif has_benign and not has_malignant:
        label = 0
    else:
        label = None

    return label, object_names, bboxes, filename_xml

def image_basic_info(path: Path):
    try:
        with Image.open(path) as im:
            return im.size[0], im.size[1], im.mode
    except Exception:
        return np.nan, np.nan, "READ_ERROR"

def print_class_counts(df, name, label_col="label"):
    print(f"\n--- {name} class distribution ---")
    print(df[label_col].value_counts(dropna=False).sort_index())
    print("Percent:")
    print((df[label_col].value_counts(dropna=False, normalize=True)
           .sort_index() * 100).round(2))

# -------------------------
# STEP 4: TN5000 split audit (reads from Drive — these are tiny txt files)
# -------------------------
print("===== TN5000 SPLIT AUDIT =====")
split_files = {
    "train":    TN5000_IMGSETS_MAIN / "train.txt",
    "val":      TN5000_IMGSETS_MAIN / "val.txt",
    "test":     TN5000_IMGSETS_MAIN / "test.txt",
    "trainval": TN5000_IMGSETS_MAIN / "trainval.txt",
}
split_ids = {k: read_split_ids(v) for k, v in split_files.items() if v.exists()}

for split, ids in split_ids.items():
    print(f"{split}: {len(ids)} IDs | unique: {len(set(ids))}")

for a, b in [("train","val"), ("train","test"), ("val","test")]:
    overlap = set(split_ids[a]).intersection(set(split_ids[b]))
    print(f"Overlap {a} vs {b}: {len(overlap)}")

trainval_union = set(split_ids["train"]).union(set(split_ids["val"]))
trainval_file  = set(split_ids.get("trainval", []))
print("train ∪ val equals trainval.txt:", trainval_union == trainval_file)

# -------------------------
# STEP 5: TN5000 XML label parsing — uses LOCAL_ANN and LOCAL_JPEG
# -------------------------
print("\n===== TN5000 XML LABEL PARSING =====")

tn_rows = []
all_object_names = Counter()
xml_parse_errors = []

for split in ["train", "val", "test"]:
    for image_id in split_ids[split]:
        img_path = LOCAL_JPEG / f"{image_id}.jpg"   # <-- local
        xml_path = LOCAL_ANN  / f"{image_id}.xml"   # <-- local
        label, object_names, bboxes, filename_xml = None, [], [], None

        if xml_path.exists():
            try:
                label, object_names, bboxes, filename_xml = infer_binary_label_from_xml(xml_path)
                all_object_names.update(object_names)
            except Exception as e:
                xml_parse_errors.append((image_id, repr(e)))
        else:
            xml_parse_errors.append((image_id, "XML_MISSING"))

        tn_rows.append({
            "dataset": "TN5000", "split": split, "image_id": image_id,
            "case_id": np.nan,
            "image_path": str(img_path),
            "xml_path":   str(xml_path),
            "label": label,
            "object_names": "|".join(object_names),
            "n_objects": len(object_names),
            "n_bboxes":  len(bboxes),
            "filename_xml": filename_xml,
            "image_exists": img_path.exists(),
            "xml_exists":   xml_path.exists(),
        })

tn5000_df = pd.DataFrame(tn_rows)

print("TN5000 manifest shape:", tn5000_df.shape)
print("Image exists:", tn5000_df["image_exists"].value_counts(dropna=False).to_dict())
print("XML exists:",   tn5000_df["xml_exists"].value_counts(dropna=False).to_dict())
print("XML parse errors:", len(xml_parse_errors))
if xml_parse_errors:
    print("First errors:", xml_parse_errors[:10])

print("\nUnique XML object names:")
for k, v in all_object_names.most_common(50):
    print(f"  {repr(k)}: {v}")

print_class_counts(tn5000_df, "TN5000 all splits")
for split in ["train", "val", "test"]:
    print_class_counts(tn5000_df[tn5000_df["split"] == split], f"TN5000 {split}")

n_missing_labels = tn5000_df["label"].isna().sum()
print("\nTN5000 missing/ambiguous labels:", n_missing_labels)
if n_missing_labels > 0:
    display(tn5000_df[tn5000_df["label"].isna()].head(20))
    print("IMPORTANT: fix label mapping before proceeding.")

# -------------------------
# STEP 6: External manifests — uses LOCAL_B1 and LOCAL_B2
# -------------------------
print("\n===== EXTERNAL DATASET MANIFEST CONSTRUCTION =====")

def build_external_manifest(batch_name, image_csv_path, label_csv_path, local_image_dir):
    image_df = pd.read_csv(image_csv_path)
    label_df = pd.read_csv(label_csv_path)

    image_df.columns = [c.strip() for c in image_df.columns]
    label_df.columns = [c.strip() for c in label_df.columns]
    image_df["patient_name"] = image_df["patient_name"].astype(str).str.strip()
    label_df["patient_name"] = label_df["patient_name"].astype(str).str.strip()
    label_df["histo_label"]  = pd.to_numeric(label_df["histo_label"], errors="coerce")

    # Build case-insensitive filename index from LOCAL cache
    image_files = list_images_case_insensitive(local_image_dir)
    file_index  = {p.name.lower(): p for p in image_files}

    merged = image_df.merge(label_df, on="patient_name", how="left", validate="many_to_one")
    rows, missing_files = [], []

    for _, r in merged.iterrows():
        rel_name = str(r["path"]).strip()
        resolved = file_index.get(Path(rel_name).name.lower(), None)
        if resolved is None:
            missing_files.append(rel_name)
        rows.append({
            "dataset": "Hou2024_external",
            "batch": batch_name,
            "split": "external",
            "case_id": str(r["patient_name"]),
            "image_id": Path(rel_name).stem,
            "image_filename_csv": rel_name,
            "image_path": str(resolved) if resolved else str(local_image_dir / rel_name),
            "label": int(r["histo_label"]) if not pd.isna(r["histo_label"]) else np.nan,
            "image_exists": resolved is not None,
        })

    out = pd.DataFrame(rows)
    print(f"\n{batch_name}")
    print(f"  CSV images: {len(image_df)} | CSV cases: {len(label_df)}")
    print(f"  Local files found: {len(image_files)}")
    print(f"  Missing image files: {(~out['image_exists']).sum()}")
    print(f"  Missing labels: {out['label'].isna().sum()}")
    if missing_files:
        print(f"  First missing: {missing_files[:10]}")
    print_class_counts(out, batch_name)
    print("Images per case:")
    print(out.groupby("case_id").size().describe())
    return out

b1_df = build_external_manifest("batch1", BATCH1_CSV, BATCH1_LABEL_CSV, LOCAL_B1)
b2_df = build_external_manifest("batch2", BATCH2_CSV, BATCH2_LABEL_CSV, LOCAL_B2)

external_df = pd.concat([b1_df, b2_df], ignore_index=True)
print("\n--- External full cohort ---")
print("Shape:", external_df.shape)
print("Unique cases:", external_df["case_id"].nunique())
print("image_exists:", external_df["image_exists"].value_counts(dropna=False).to_dict())
print_class_counts(external_df, "External full cohort")

inconsistent = (external_df.groupby(["batch","case_id"])["label"]
                .nunique(dropna=False))
inconsistent = inconsistent[inconsistent > 1]
print("Inconsistent case labels:", len(inconsistent))

# -------------------------
# STEP 7: Quick image readability sample — all local, fast
# -------------------------
print("\n===== QUICK IMAGE READABILITY SAMPLE =====")
sample_rows = []
for name, df in [
    ("TN5000_train",   tn5000_df[tn5000_df["split"]=="train"].sample(20, random_state=42)),
    ("External_batch1", b1_df.sample(min(20, len(b1_df)), random_state=42)),
    ("External_batch2", b2_df.sample(min(20, len(b2_df)), random_state=42)),
]:
    for _, r in df.iterrows():
        w, h, mode = image_basic_info(Path(r["image_path"]))
        sample_rows.append({"group": name, "width": w, "height": h,
                             "mode": mode, "label": r["label"]})

sample_df = pd.DataFrame(sample_rows)
display(sample_df.groupby(["group","mode"]).size().reset_index(name="count"))
display(sample_df.groupby("group")[["width","height"]].describe())

# -------------------------
# STEP 8: Save everything
# -------------------------
tn5000_df.to_csv(   OUT_ROOT / "manifest_tn5000_locked_splits.csv", index=False)
external_df.to_csv( OUT_ROOT / "manifest_hou2024_external.csv",     index=False)
sample_df.to_csv(   OUT_ROOT / "cell2_image_sample.csv",            index=False)

summary = {
    "tn5000_rows": len(tn5000_df),
    "tn5000_train": int((tn5000_df["split"]=="train").sum()),
    "tn5000_val":   int((tn5000_df["split"]=="val").sum()),
    "tn5000_test":  int((tn5000_df["split"]=="test").sum()),
    "tn5000_missing_labels": int(tn5000_df["label"].isna().sum()),
    "tn5000_missing_images": int((~tn5000_df["image_exists"]).sum()),
    "tn5000_missing_xml":    int((~tn5000_df["xml_exists"]).sum()),
    "external_rows":          len(external_df),
    "external_cases":         int(external_df["case_id"].nunique()),
    "external_missing_labels":int(external_df["label"].isna().sum()),
    "external_missing_images":int((~external_df["image_exists"]).sum()),
    "batch1_rows":  int((external_df["batch"]=="batch1").sum()),
    "batch2_rows":  int((external_df["batch"]=="batch2").sum()),
    "batch1_cases": int(external_df.loc[external_df["batch"]=="batch1","case_id"].nunique()),
    "batch2_cases": int(external_df.loc[external_df["batch"]=="batch2","case_id"].nunique()),
}
with open(OUT_ROOT / "cell2_manifest_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n===== CELL 2 SUMMARY =====")
print(json.dumps(summary, indent=2))
print("\nCELL 2 COMPLETE. Paste the full output here before running the next cell.")

=== CACHING TO LOCAL DISK (one-time, ~2-5 min) ===
  Copying 5000 TN5000 XMLs files to local disk...
  Done in 128.9s
  Copying 5000 TN5000 JPEGs files to local disk...
  Done in 117.1s
  Copying 6005 Batch1 images files to local disk...
  Done in 176.6s
  Copying 2503 Batch2 images files to local disk...
  Done in 94.2s
All cached.

===== TN5000 SPLIT AUDIT =====
train: 3500 IDs | unique: 3500
val: 500 IDs | unique: 500
test: 1000 IDs | unique: 1000
trainval: 4000 IDs | unique: 4000
Overlap train vs val: 0
Overlap train vs test: 0
Overlap val vs test: 0
train ∪ val equals trainval.txt: True

===== TN5000 XML LABEL PARSING =====
TN5000 manifest shape: (5000, 13)
Image exists: {True: 5000}
XML exists: {True: 5000}
XML parse errors: 0

Unique XML object names:
  '1': 3574
  '0': 1426

--- TN5000 all splits class distribution ---
label
0    1426
1    3574
Name: count, dtype: int64
Percent:
label
0    28.52
1    71.48
Name: proportion, dtype: float64

--- TN5000 train class distribution --

,group,mode,count
0,External_batch1,READ_ERROR,6
1,External_batch1,RGB,14
2,External_batch2,RGB,20
3,TN5000_train,RGB,20


width                                                       \
                count    mean         std    min      25%     50%      75%   
group                                                                        
External_batch1  14.0   809.5  178.849206  548.0   688.25   804.0   965.75   
External_batch2  20.0  1117.2  127.400570  960.0  1024.00  1024.0  1260.00   
TN5000_train     20.0   712.0   55.097999  498.0   718.00   718.0   718.00   

                        height                                               \
                    max  count        mean        std    min     25%    50%   
group                                                                         
External_batch1  1058.0   14.0  588.357143  56.888024  498.0  541.25  603.0   
External_batch2  1292.0   20.0  824.400000  87.382041  708.0  768.00  768.0   
TN5000_train      818.0   20.0  502.850000  33.452677  429.0  500.00  500.0   

                               
                   75%    max  
group                          
External_batch1  636.0  660.0  
External_batch2  910.0  970.0  
TN5000_train     500.0  628.0


===== CELL 2 SUMMARY =====
{
  "tn5000_rows": 5000,
  "tn5000_train": 3500,
  "tn5000_val": 500,
  "tn5000_test": 1000,
  "tn5000_missing_labels": 0,
  "tn5000_missing_images": 0,
  "tn5000_missing_xml": 0,
  "external_rows": 8508,
  "external_cases": 601,
  "external_missing_labels": 8,
  "external_missing_images": 2607,
  "batch1_rows": 6005,
  "batch2_rows": 2503,
  "batch1_cases": 601,
  "batch2_cases": 242
}

CELL 2 COMPLETE. Paste the full output here before running the next cell.


In [ ]:
# =========================
# CELL 3 — Repair external manifest path resolution + case ID collision audit
# =========================

import os, re, json, math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image

# -------------------------
# Paths
# -------------------------
DRIVE_ROOT = Path("/content/drive/MyDrive")

EXT_ROOT = DRIVE_ROOT / "2"
BATCH1_ROOT = EXT_ROOT / "batch1_image"
BATCH1_DATASET = BATCH1_ROOT / "dataset"
BATCH1_CSV = BATCH1_ROOT / "batch1_image.csv"
BATCH1_LABEL_CSV = BATCH1_ROOT / "batch1_image_label.csv"

BATCH2_ROOT = EXT_ROOT / "batch2_image"
BATCH2_DATASET = BATCH2_ROOT / "dataset"
BATCH2_CSV = BATCH2_ROOT / "batch2_image.csv"
BATCH2_LABEL_CSV = BATCH2_ROOT / "batch2_image_label.csv"

OUT_ROOT = DRIVE_ROOT / "thyroid_external_validation_outputs"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_B1 = Path("/tmp/batch1")
LOCAL_B2 = Path("/tmp/batch2")

VALID_IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# -------------------------
# Helpers
# -------------------------
def list_images_case_insensitive(root: Path):
    if not root.exists():
        return []
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VALID_IMAGE_SUFFIXES])

def clean_patient(x):
    # Make patient IDs stable across int/float/string forms.
    s = str(x).strip()
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    return s

def normalize_filename_key(name):
    """
    Conservative normalization for file matching:
    lower-case, remove extension, remove non-alphanumeric characters.
    """
    stem = Path(str(name).strip()).stem.lower()
    return re.sub(r"[^a-z0-9]+", "", stem)

def parse_patient_seq(filename):
    """
    Extract patient and sequence from names like:
      341_007.Jpg
      195_001_173546.Jpg
    """
    stem = Path(str(filename).strip()).stem
    parts = stem.split("_")
    if len(parts) >= 2 and parts[0].isdigit() and parts[1].isdigit():
        return parts[0], parts[1]
    return None, None

def image_basic_info_safe(path):
    try:
        with Image.open(path) as im:
            return im.size[0], im.size[1], im.mode, True
    except Exception:
        return np.nan, np.nan, "READ_ERROR", False

def summarize_label_counts(df, name):
    print(f"\n--- {name} label distribution, image-level ---")
    print(df["label"].value_counts(dropna=False).sort_index())
    print("Percent:")
    print((df["label"].value_counts(dropna=False, normalize=True).sort_index() * 100).round(2))

    case_df = df.dropna(subset=["label"]).drop_duplicates("case_uid")
    print(f"\n--- {name} label distribution, case-level ---")
    print(case_df["label"].value_counts(dropna=False).sort_index())
    print("Percent:")
    print((case_df["label"].value_counts(dropna=False, normalize=True).sort_index() * 100).round(2))

def build_file_indices(image_dir: Path):
    files = list_images_case_insensitive(image_dir)

    exact = {}
    normalized = defaultdict(list)
    patient_seq = defaultdict(list)
    by_patient = defaultdict(list)

    duplicate_exact_names = []

    for p in files:
        key = p.name.lower().strip()
        if key in exact:
            duplicate_exact_names.append(key)
        exact[key] = p

        norm_key = normalize_filename_key(p.name)
        normalized[norm_key].append(p)

        patient, seq = parse_patient_seq(p.name)
        if patient is not None:
            patient_seq[(patient, seq)].append(p)
            by_patient[patient].append(p)

    return {
        "files": files,
        "exact": exact,
        "normalized": normalized,
        "patient_seq": patient_seq,
        "by_patient": by_patient,
        "duplicate_exact_names": duplicate_exact_names,
    }

def resolve_one_filename(csv_filename, indices):
    """
    Return resolved_path, method, candidate_count.
    Methods:
      exact_lower
      normalized_unique
      patient_seq_unique
      unresolved
    """
    csv_name = Path(str(csv_filename).strip()).name
    lower_key = csv_name.lower().strip()

    # 1. Exact case-insensitive basename match
    if lower_key in indices["exact"]:
        return indices["exact"][lower_key], "exact_lower", 1

    # 2. Normalized unique match
    norm_key = normalize_filename_key(csv_name)
    norm_candidates = indices["normalized"].get(norm_key, [])
    if len(norm_candidates) == 1:
        return norm_candidates[0], "normalized_unique", 1
    elif len(norm_candidates) > 1:
        return None, "normalized_ambiguous", len(norm_candidates)

    # 3. Patient + sequence unique match
    patient, seq = parse_patient_seq(csv_name)
    if patient is not None:
        ps_candidates = indices["patient_seq"].get((patient, seq), [])
        if len(ps_candidates) == 1:
            return ps_candidates[0], "patient_seq_unique", 1
        elif len(ps_candidates) > 1:
            return None, "patient_seq_ambiguous", len(ps_candidates)

    return None, "unresolved", 0

def build_external_manifest_repaired(batch_name, image_csv_path, label_csv_path, local_image_dir):
    print(f"\n==================== {batch_name.upper()} REPAIR ====================")

    image_df = pd.read_csv(image_csv_path)
    label_df = pd.read_csv(label_csv_path)

    image_df.columns = [c.strip() for c in image_df.columns]
    label_df.columns = [c.strip() for c in label_df.columns]

    image_df["patient_name"] = image_df["patient_name"].apply(clean_patient)
    label_df["patient_name"] = label_df["patient_name"].apply(clean_patient)
    label_df["histo_label"] = pd.to_numeric(label_df["histo_label"], errors="coerce")

    indices = build_file_indices(local_image_dir)

    print("Image CSV rows:", len(image_df))
    print("Label CSV rows:", len(label_df))
    print("Unique image-CSV patients:", image_df["patient_name"].nunique())
    print("Unique label-CSV patients:", label_df["patient_name"].nunique())
    print("Local image files discovered:", len(indices["files"]))
    print("Duplicate exact local basenames:", len(indices["duplicate_exact_names"]))
    if indices["duplicate_exact_names"][:10]:
        print("Example duplicate basenames:", indices["duplicate_exact_names"][:10])

    merged = image_df.merge(label_df, on="patient_name", how="left", validate="many_to_one")

    rows = []
    method_counter = Counter()
    unresolved_examples = []
    ambiguous_examples = []

    for _, r in merged.iterrows():
        csv_filename = str(r["path"]).strip()
        resolved_path, method, candidate_count = resolve_one_filename(csv_filename, indices)
        method_counter[method] += 1

        if method == "unresolved" and len(unresolved_examples) < 30:
            patient, seq = parse_patient_seq(csv_filename)
            patient_candidates = indices["by_patient"].get(patient, [])[:15] if patient is not None else []
            unresolved_examples.append({
                "csv_filename": csv_filename,
                "patient": patient,
                "seq": seq,
                "first_patient_candidates": [p.name for p in patient_candidates]
            })

        if "ambiguous" in method and len(ambiguous_examples) < 30:
            ambiguous_examples.append({
                "csv_filename": csv_filename,
                "method": method,
                "candidate_count": candidate_count
            })

        label_value = r["histo_label"]
        label_value = int(label_value) if not pd.isna(label_value) else np.nan

        rows.append({
            "dataset": "Hou2024_external",
            "batch": batch_name,
            "split": "external",
            "patient_name": str(r["patient_name"]),
            "case_uid": f"{batch_name}:{r['patient_name']}",
            "image_id": Path(csv_filename).stem,
            "image_filename_csv": csv_filename,
            "image_path": str(resolved_path) if resolved_path is not None else str(local_image_dir / csv_filename),
            "path_resolution_method": method,
            "path_candidate_count": int(candidate_count),
            "label": label_value,
            "image_exists": resolved_path is not None,
        })

    out = pd.DataFrame(rows)

    print("\nPath resolution methods:")
    print(dict(method_counter))

    print("Missing image files after repair:", int((~out["image_exists"]).sum()))
    print("Missing labels:", int(out["label"].isna().sum()))

    if unresolved_examples:
        print("\nUnresolved examples with local candidates from same patient:")
        for ex in unresolved_examples[:10]:
            print(json.dumps(ex, indent=2))

    if ambiguous_examples:
        print("\nAmbiguous examples:")
        for ex in ambiguous_examples[:10]:
            print(json.dumps(ex, indent=2))

    missing_label_cases = out.loc[out["label"].isna(), ["batch", "patient_name", "case_uid", "image_filename_csv"]].drop_duplicates()
    if len(missing_label_cases) > 0:
        print("\nRows/cases with missing labels:")
        display(missing_label_cases.head(50))

    # Readability check on resolved files
    resolved_sample = out[out["image_exists"]].sample(min(30, out["image_exists"].sum()), random_state=42)
    read_rows = []
    for _, rr in resolved_sample.iterrows():
        w, h, mode, ok = image_basic_info_safe(Path(rr["image_path"]))
        read_rows.append({
            "batch": batch_name,
            "image_path": rr["image_path"],
            "width": w,
            "height": h,
            "mode": mode,
            "read_ok": ok
        })
    read_df = pd.DataFrame(read_rows)
    print("\nReadability sample:")
    print(read_df["read_ok"].value_counts(dropna=False).to_dict())
    print(read_df["mode"].value_counts(dropna=False).to_dict())

    summarize_label_counts(out, batch_name)

    return out, read_df

# -------------------------
# Build repaired manifests
# -------------------------
b1_repaired, b1_read = build_external_manifest_repaired(
    "batch1", BATCH1_CSV, BATCH1_LABEL_CSV, LOCAL_B1
)

b2_repaired, b2_read = build_external_manifest_repaired(
    "batch2", BATCH2_CSV, BATCH2_LABEL_CSV, LOCAL_B2
)

external_raw = pd.concat([b1_repaired, b2_repaired], ignore_index=True)
readability_sample = pd.concat([b1_read, b2_read], ignore_index=True)

print("\n==================== FULL EXTERNAL RAW SUMMARY ====================")
print("Rows:", len(external_raw))
print("Unique case_uid:", external_raw["case_uid"].nunique())
print("Unique raw patient_name without batch prefix:", external_raw["patient_name"].nunique())
print("Image exists:")
print(external_raw["image_exists"].value_counts(dropna=False))
print("Missing labels:", int(external_raw["label"].isna().sum()))

summarize_label_counts(external_raw, "Full external raw")

# -------------------------
# Build final usable external manifest
# -------------------------
# For primary evaluation, exclude rows with unresolved image path or missing label.
# This is a data-quality exclusion, not model tuning.
external_usable = external_raw[(external_raw["image_exists"]) & (~external_raw["label"].isna())].copy()
external_usable["label"] = external_usable["label"].astype(int)

print("\n==================== FULL EXTERNAL USABLE SUMMARY ====================")
print("Usable rows:", len(external_usable))
print("Excluded rows:", len(external_raw) - len(external_usable))
print("Usable unique case_uid:", external_usable["case_uid"].nunique())
print("Usable cases by batch:")
print(external_usable.groupby("batch")["case_uid"].nunique())
print("Usable images by batch:")
print(external_usable["batch"].value_counts())

summarize_label_counts(external_usable, "Full external usable")

# Check case-level label consistency
case_label_nunique = external_usable.groupby("case_uid")["label"].nunique(dropna=False)
bad_cases = case_label_nunique[case_label_nunique > 1]
print("\nCase-level inconsistent labels after repair:", len(bad_cases))
if len(bad_cases) > 0:
    display(bad_cases.head(20))

# Check duplicate image paths
dup_paths = external_usable["image_path"].duplicated().sum()
print("Duplicate image_path rows in usable external manifest:", int(dup_paths))
if dup_paths > 0:
    display(external_usable[external_usable["image_path"].duplicated(keep=False)].head(30))

# -------------------------
# Save outputs
# -------------------------
raw_path = OUT_ROOT / "manifest_hou2024_external_raw_repaired.csv"
usable_path = OUT_ROOT / "manifest_hou2024_external_usable_repaired.csv"
read_path = OUT_ROOT / "cell3_external_readability_sample.csv"
summary_path = OUT_ROOT / "cell3_external_manifest_repair_summary.json"

external_raw.to_csv(raw_path, index=False)
external_usable.to_csv(usable_path, index=False)
readability_sample.to_csv(read_path, index=False)

summary = {
    "external_raw_rows": int(len(external_raw)),
    "external_raw_case_uid": int(external_raw["case_uid"].nunique()),
    "external_raw_unique_patient_name_without_batch_prefix": int(external_raw["patient_name"].nunique()),
    "external_raw_missing_images": int((~external_raw["image_exists"]).sum()),
    "external_raw_missing_labels": int(external_raw["label"].isna().sum()),
    "external_usable_rows": int(len(external_usable)),
    "external_usable_case_uid": int(external_usable["case_uid"].nunique()),
    "external_usable_excluded_rows": int(len(external_raw) - len(external_usable)),
    "external_usable_batch1_rows": int((external_usable["batch"] == "batch1").sum()),
    "external_usable_batch2_rows": int((external_usable["batch"] == "batch2").sum()),
    "external_usable_batch1_cases": int(external_usable.loc[external_usable["batch"] == "batch1", "case_uid"].nunique()),
    "external_usable_batch2_cases": int(external_usable.loc[external_usable["batch"] == "batch2", "case_uid"].nunique()),
    "external_usable_duplicate_image_paths": int(dup_paths),
    "case_label_inconsistent_cases": int(len(bad_cases)),
    "batch1_resolution_methods": dict(Counter(b1_repaired["path_resolution_method"])),
    "batch2_resolution_methods": dict(Counter(b2_repaired["path_resolution_method"])),
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n==================== SAVED ====================")
print(raw_path)
print(usable_path)
print(read_path)
print(summary_path)

print("\n==================== CELL 3 SUMMARY ====================")
print(json.dumps(summary, indent=2))

print("\nCELL 3 COMPLETE. Paste the full output here before running the next cell.")


==================== BATCH1 REPAIR ====================
Image CSV rows: 6005
Label CSV rows: 601
Unique image-CSV patients: 601
Unique label-CSV patients: 601
Local image files discovered: 6005
Duplicate exact local basenames: 0

Path resolution methods:
{'exact_lower': 3398, 'unresolved': 2607}
Missing image files after repair: 2607
Missing labels: 0

Unresolved examples with local candidates from same patient:
{
  "csv_filename": "341_007.Jpg",
  "patient": "341",
  "seq": "007",
  "first_patient_candidates": [
    "341_001.Jpg",
    "341_002.Jpg",
    "341_003.Jpg",
    "341_004.Jpg",
    "341_005.Jpg",
    "341_006.Jpg",
    "341_008.Jpg",
    "341_009.Jpg",
    "341_010.Jpg",
    "341_011.Jpg"
  ]
}
{
  "csv_filename": "239_001.Jpg",
  "patient": "239",
  "seq": "001",
  "first_patient_candidates": [
    "239_027.Jpg",
    "239_028.Jpg",
    "239_029.Jpg",
    "239_030.Jpg",
    "239_031.Jpg",
    "239_032.Jpg",
    "239_033.Jpg",
    "239_034.Jpg",
    "239_035.Jpg",
    "239_03

,batch,patient_name,case_uid,image_filename_csv
2183,batch2,200,batch2:200,200_001_180125.Jpg
2184,batch2,200,batch2:200,200_002_180125.Jpg
2185,batch2,200,batch2:200,200_003_180125.Jpg
2186,batch2,200,batch2:200,200_004_180125.Jpg
2187,batch2,200,batch2:200,200_005_180126.Jpg
2188,batch2,200,batch2:200,200_006_180126.Jpg
2189,batch2,200,batch2:200,200_007_180126.Jpg
2190,batch2,200,batch2:200,200_008_180126.Jpg



Readability sample:
{True: 30}
{'RGB': 30}

--- batch2 label distribution, image-level ---
label
0.0     961
1.0    1534
NaN       8
Name: count, dtype: int64
Percent:
label
0.0    38.39
1.0    61.29
NaN     0.32
Name: proportion, dtype: float64

--- batch2 label distribution, case-level ---
label
0.0    100
1.0    141
Name: count, dtype: int64
Percent:
label
0.0    41.49
1.0    58.51
Name: proportion, dtype: float64

==================== FULL EXTERNAL RAW SUMMARY ====================
Rows: 8508
Unique case_uid: 843
Unique raw patient_name without batch prefix: 601
Image exists:
image_exists
True     5901
False    2607
Name: count, dtype: int64
Missing labels: 8

--- Full external raw label distribution, image-level ---
label
0.0    2995
1.0    5505
NaN       8
Name: count, dtype: int64
Percent:
label
0.0    35.20
1.0    64.70
NaN     0.09
Name: proportion, dtype: float64

--- Full external raw label distribution, case-level ---
label
0.0    318
1.0    524
Name: count, dtype: int64
Pe

In [ ]:
# =========================
# CELL 4 — Diagnose Batch 1 CSV-folder mismatch and build safer actual-file external manifest
# =========================

import os, re, json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image

# -------------------------
# Paths
# -------------------------
DRIVE_ROOT = Path("/content/drive/MyDrive")

EXT_ROOT = DRIVE_ROOT / "2"
BATCH1_ROOT = EXT_ROOT / "batch1_image"
BATCH1_CSV = BATCH1_ROOT / "batch1_image.csv"
BATCH1_LABEL_CSV = BATCH1_ROOT / "batch1_image_label.csv"

BATCH2_ROOT = EXT_ROOT / "batch2_image"
BATCH2_CSV = BATCH2_ROOT / "batch2_image.csv"
BATCH2_LABEL_CSV = BATCH2_ROOT / "batch2_image_label.csv"

OUT_ROOT = DRIVE_ROOT / "thyroid_external_validation_outputs"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_B1 = Path("/tmp/batch1")
LOCAL_B2 = Path("/tmp/batch2")

VALID_IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# -------------------------
# Helpers
# -------------------------
def list_images_case_insensitive(root: Path):
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VALID_IMAGE_SUFFIXES])

def clean_patient(x):
    s = str(x).strip()
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    return s

def parse_patient_from_filename(filename):
    """
    Conservative: image names begin with patient ID before first underscore.
    Examples:
      341_001.Jpg -> 341
      195_001_173546.Jpg -> 195
    """
    stem = Path(str(filename)).stem
    first = stem.split("_")[0]
    if first.isdigit():
        return first
    return None

def image_readable(path):
    try:
        with Image.open(path) as im:
            return True, im.size[0], im.size[1], im.mode
    except Exception:
        return False, np.nan, np.nan, "READ_ERROR"

def summarize_manifest(df, name):
    print(f"\n==================== {name} ====================")
    print("Rows/images:", len(df))
    print("Unique case_uid:", df["case_uid"].nunique())
    print("Missing labels:", int(df["label"].isna().sum()))
    print("Image exists:", df["image_exists"].value_counts(dropna=False).to_dict())
    print("\nImage-level label distribution:")
    print(df["label"].value_counts(dropna=False).sort_index())
    print((df["label"].value_counts(dropna=False, normalize=True).sort_index() * 100).round(2))
    case_df = df.dropna(subset=["label"]).drop_duplicates("case_uid")
    print("\nCase-level label distribution:")
    print(case_df["label"].value_counts(dropna=False).sort_index())
    print((case_df["label"].value_counts(dropna=False, normalize=True).sort_index() * 100).round(2))
    print("\nImages per case summary:")
    print(df.groupby("case_uid").size().describe())

def build_actual_file_manifest(batch_name, local_dir, label_csv_path):
    labels = pd.read_csv(label_csv_path)
    labels.columns = [c.strip() for c in labels.columns]
    labels["patient_name"] = labels["patient_name"].apply(clean_patient)
    labels["histo_label"] = pd.to_numeric(labels["histo_label"], errors="coerce")

    label_map = dict(zip(labels["patient_name"], labels["histo_label"]))

    image_files = list_images_case_insensitive(local_dir)

    rows = []
    parse_fail = []

    for p in image_files:
        patient = parse_patient_from_filename(p.name)
        if patient is None:
            parse_fail.append(p.name)

        label = label_map.get(patient, np.nan)
        rows.append({
            "dataset": "Hou2024_external",
            "batch": batch_name,
            "split": "external",
            "patient_name": patient,
            "case_uid": f"{batch_name}:{patient}" if patient is not None else np.nan,
            "image_id": p.stem,
            "image_filename_actual": p.name,
            "image_filename_csv": np.nan,
            "image_path": str(p),
            "path_source": "actual_file_list",
            "label": int(label) if not pd.isna(label) else np.nan,
            "image_exists": p.exists(),
        })

    out = pd.DataFrame(rows)

    print(f"\n{batch_name}: actual-file manifest")
    print("Actual image files:", len(image_files))
    print("Patient parse failures:", len(parse_fail))
    if parse_fail[:20]:
        print("First parse failures:", parse_fail[:20])
    print("Unique patients parsed from files:", out["patient_name"].nunique(dropna=True))
    print("Label CSV patients:", labels["patient_name"].nunique())
    print("Patients in files but not labels:", len(set(out["patient_name"].dropna()) - set(labels["patient_name"])))
    print("Patients in labels but not files:", len(set(labels["patient_name"]) - set(out["patient_name"].dropna())))
    if len(set(out["patient_name"].dropna()) - set(labels["patient_name"])) > 0:
        print("Examples files not labels:", sorted(list(set(out["patient_name"].dropna()) - set(labels["patient_name"])))[:30])
    if len(set(labels["patient_name"]) - set(out["patient_name"].dropna())) > 0:
        print("Examples labels not files:", sorted(list(set(labels["patient_name"]) - set(out["patient_name"].dropna())))[:30])

    return out

def build_csv_manifest_for_comparison(batch_name, image_csv_path, label_csv_path, local_dir):
    imgdf = pd.read_csv(image_csv_path)
    lbldf = pd.read_csv(label_csv_path)
    imgdf.columns = [c.strip() for c in imgdf.columns]
    lbldf.columns = [c.strip() for c in lbldf.columns]
    imgdf["patient_name"] = imgdf["patient_name"].apply(clean_patient)
    lbldf["patient_name"] = lbldf["patient_name"].apply(clean_patient)
    lbldf["histo_label"] = pd.to_numeric(lbldf["histo_label"], errors="coerce")

    file_index = {p.name.lower(): p for p in list_images_case_insensitive(local_dir)}
    merged = imgdf.merge(lbldf, on="patient_name", how="left", validate="many_to_one")

    rows = []
    for _, r in merged.iterrows():
        fname = str(r["path"]).strip()
        resolved = file_index.get(Path(fname).name.lower(), None)
        rows.append({
            "batch": batch_name,
            "patient_name": r["patient_name"],
            "case_uid": f"{batch_name}:{r['patient_name']}",
            "image_filename_csv": fname,
            "csv_file_exists": resolved is not None,
            "label": int(r["histo_label"]) if not pd.isna(r["histo_label"]) else np.nan,
        })

    return pd.DataFrame(rows)

# -------------------------
# Build actual-file manifests
# -------------------------
b1_actual = build_actual_file_manifest("batch1", LOCAL_B1, BATCH1_LABEL_CSV)
b2_actual = build_actual_file_manifest("batch2", LOCAL_B2, BATCH2_LABEL_CSV)

summarize_manifest(b1_actual, "Batch 1 actual-file manifest")
summarize_manifest(b2_actual, "Batch 2 actual-file manifest")

# -------------------------
# Compare Batch 1 CSV list with actual folder list
# -------------------------
print("\n==================== BATCH 1 CSV VS ACTUAL FILE LIST ====================")

b1_csv_compare = build_csv_manifest_for_comparison("batch1", BATCH1_CSV, BATCH1_LABEL_CSV, LOCAL_B1)

b1_csv_names = set(b1_csv_compare["image_filename_csv"].astype(str).str.lower())
b1_actual_names = set(b1_actual["image_filename_actual"].astype(str).str.lower())

csv_not_actual = sorted(list(b1_csv_names - b1_actual_names))
actual_not_csv = sorted(list(b1_actual_names - b1_csv_names))

print("CSV rows:", len(b1_csv_compare))
print("Actual files:", len(b1_actual))
print("CSV names not in actual folder:", len(csv_not_actual))
print("Actual filenames not in CSV:", len(actual_not_csv))
print("First 30 CSV not actual:", csv_not_actual[:30])
print("First 30 actual not CSV:", actual_not_csv[:30])

# Patient-level comparison of image counts
b1_csv_counts = b1_csv_compare.groupby("patient_name").size().rename("csv_n")
b1_actual_counts = b1_actual.groupby("patient_name").size().rename("actual_n")
count_compare = pd.concat([b1_csv_counts, b1_actual_counts], axis=1).fillna(0).astype(int)
count_compare["diff_actual_minus_csv"] = count_compare["actual_n"] - count_compare["csv_n"]
count_compare = count_compare.sort_values("diff_actual_minus_csv")

print("\nBatch 1 patient count agreement:")
print("Patients with same count:", int((count_compare["diff_actual_minus_csv"] == 0).sum()))
print("Patients with different count:", int((count_compare["diff_actual_minus_csv"] != 0).sum()))
print("\nMost negative differences:")
display(count_compare.head(20))
print("\nMost positive differences:")
display(count_compare.tail(20))

# -------------------------
# Readability samples and full readability check only for actual external files
# -------------------------
print("\n==================== READABILITY CHECK, ACTUAL EXTERNAL FILES ====================")

external_actual_raw = pd.concat([b1_actual, b2_actual], ignore_index=True)

read_rows = []
# Full external has 8508 files; checking all is useful before training and should be acceptable.
for i, r in external_actual_raw.iterrows():
    ok, w, h, mode = image_readable(Path(r["image_path"]))
    read_rows.append({
        "batch": r["batch"],
        "case_uid": r["case_uid"],
        "image_path": r["image_path"],
        "read_ok": ok,
        "width": w,
        "height": h,
        "mode": mode,
        "label": r["label"]
    })

external_read_df = pd.DataFrame(read_rows)
print("Readability:")
print(external_read_df["read_ok"].value_counts(dropna=False))
print("\nMode by batch:")
print(pd.crosstab(external_read_df["batch"], external_read_df["mode"]))
print("\nDimension summary by batch:")
display(external_read_df.groupby("batch")[["width", "height"]].describe())

unreadable = external_read_df[~external_read_df["read_ok"]]
if len(unreadable) > 0:
    print("\nUnreadable files:")
    display(unreadable.head(50))

# -------------------------
# Final actual-file external usable manifest
# -------------------------
external_actual_usable = external_actual_raw[
    external_actual_raw["image_exists"] &
    (~external_actual_raw["label"].isna()) &
    (external_actual_raw["case_uid"].notna())
].copy()
external_actual_usable["label"] = external_actual_usable["label"].astype(int)

# Exclude unreadable if any
unreadable_paths = set(external_read_df.loc[~external_read_df["read_ok"], "image_path"])
if len(unreadable_paths) > 0:
    external_actual_usable = external_actual_usable[~external_actual_usable["image_path"].isin(unreadable_paths)].copy()

print("\n==================== FINAL EXTERNAL ACTUAL-FILE USABLE MANIFEST ====================")
summarize_manifest(external_actual_usable, "Full external actual-file usable")

print("\nUsable images by batch:")
print(external_actual_usable["batch"].value_counts())
print("\nUsable cases by batch:")
print(external_actual_usable.groupby("batch")["case_uid"].nunique())

print("\nMissing-label cases in actual-file raw manifest:")
missing_label_cases = external_actual_raw[external_actual_raw["label"].isna()][["batch", "patient_name", "case_uid"]].drop_duplicates()
display(missing_label_cases)

# Expected external final should be 8500 images / 842 cases if only batch2 patient 200 lacks labels.
# Confirm this explicitly.
print("\nExpected-paper-aligned check:")
print("Raw actual images:", len(external_actual_raw))
print("Raw actual cases:", external_actual_raw["case_uid"].nunique())
print("Usable images:", len(external_actual_usable))
print("Usable cases:", external_actual_usable["case_uid"].nunique())

# Duplicate paths and inconsistent labels
dup_paths = external_actual_usable["image_path"].duplicated().sum()
case_label_nunique = external_actual_usable.groupby("case_uid")["label"].nunique(dropna=False)
bad_cases = case_label_nunique[case_label_nunique > 1]

print("Duplicate image paths:", int(dup_paths))
print("Cases with inconsistent labels:", int(len(bad_cases)))

# -------------------------
# Save outputs
# -------------------------
b1_actual_path = OUT_ROOT / "manifest_batch1_actual_file_based.csv"
b2_actual_path = OUT_ROOT / "manifest_batch2_actual_file_based.csv"
external_actual_raw_path = OUT_ROOT / "manifest_hou2024_external_actual_file_raw.csv"
external_actual_usable_path = OUT_ROOT / "manifest_hou2024_external_actual_file_usable.csv"
readability_path = OUT_ROOT / "cell4_external_actual_readability.csv"
count_compare_path = OUT_ROOT / "cell4_batch1_csv_vs_actual_patient_counts.csv"
summary_path = OUT_ROOT / "cell4_external_actual_manifest_summary.json"

b1_actual.to_csv(b1_actual_path, index=False)
b2_actual.to_csv(b2_actual_path, index=False)
external_actual_raw.to_csv(external_actual_raw_path, index=False)
external_actual_usable.to_csv(external_actual_usable_path, index=False)
external_read_df.to_csv(readability_path, index=False)
count_compare.to_csv(count_compare_path)

summary = {
    "batch1_actual_images": int(len(b1_actual)),
    "batch1_actual_cases": int(b1_actual["case_uid"].nunique()),
    "batch1_actual_missing_labels": int(b1_actual["label"].isna().sum()),
    "batch1_csv_names_not_in_actual": int(len(csv_not_actual)),
    "batch1_actual_names_not_in_csv": int(len(actual_not_csv)),
    "batch1_patients_same_count_csv_vs_actual": int((count_compare["diff_actual_minus_csv"] == 0).sum()),
    "batch1_patients_different_count_csv_vs_actual": int((count_compare["diff_actual_minus_csv"] != 0).sum()),
    "batch2_actual_images": int(len(b2_actual)),
    "batch2_actual_cases": int(b2_actual["case_uid"].nunique()),
    "batch2_actual_missing_labels": int(b2_actual["label"].isna().sum()),
    "external_actual_raw_images": int(len(external_actual_raw)),
    "external_actual_raw_cases": int(external_actual_raw["case_uid"].nunique()),
    "external_actual_raw_missing_labels": int(external_actual_raw["label"].isna().sum()),
    "external_actual_usable_images": int(len(external_actual_usable)),
    "external_actual_usable_cases": int(external_actual_usable["case_uid"].nunique()),
    "external_actual_usable_batch1_images": int((external_actual_usable["batch"] == "batch1").sum()),
    "external_actual_usable_batch2_images": int((external_actual_usable["batch"] == "batch2").sum()),
    "external_actual_usable_batch1_cases": int(external_actual_usable.loc[external_actual_usable["batch"] == "batch1", "case_uid"].nunique()),
    "external_actual_usable_batch2_cases": int(external_actual_usable.loc[external_actual_usable["batch"] == "batch2", "case_uid"].nunique()),
    "external_unreadable_files": int((~external_read_df["read_ok"]).sum()),
    "external_duplicate_image_paths": int(dup_paths),
    "external_inconsistent_case_labels": int(len(bad_cases)),
    "missing_label_cases": missing_label_cases.to_dict(orient="records"),
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n==================== SAVED ====================")
for p in [
    b1_actual_path,
    b2_actual_path,
    external_actual_raw_path,
    external_actual_usable_path,
    readability_path,
    count_compare_path,
    summary_path,
]:
    print(p)

print("\n==================== CELL 4 SUMMARY ====================")
print(json.dumps(summary, indent=2))

print("\nCELL 4 COMPLETE. Paste the full output here before running the next cell.")


batch1: actual-file manifest
Actual image files: 6005
Patient parse failures: 0
Unique patients parsed from files: 601
Label CSV patients: 601
Patients in files but not labels: 0
Patients in labels but not files: 0

batch2: actual-file manifest
Actual image files: 2503
Patient parse failures: 0
Unique patients parsed from files: 242
Label CSV patients: 241
Patients in files but not labels: 1
Patients in labels but not files: 0
Examples files not labels: ['200']

==================== Batch 1 actual-file manifest ====================
Rows/images: 6005
Unique case_uid: 601
Missing labels: 0
Image exists: {True: 6005}

Image-level label distribution:
label
0    2136
1    3869
Name: count, dtype: int64
label
0    35.57
1    64.43
Name: proportion, dtype: float64

Case-level label distribution:
label
0    218
1    383
Name: count, dtype: int64
label
0    36.27
1    63.73
Name: proportion, dtype: float64

Images per case summary:
count    601.000000
mean       9.991681
std        4.877663
mi

,csv_n,actual_n,diff_actual_minus_csv
patient_name,,,
578,29,2,-27
44,28,4,-24
304,27,4,-23
149,25,5,-20
180,23,3,-20
593,31,11,-20
197,21,2,-19
219,21,5,-16
148,19,3,-16



Most positive differences:


,csv_n,actual_n,diff_actual_minus_csv
patient_name,,,
424,4,16,12
113,3,15,12
152,10,22,12
122,8,20,12
526,11,24,13
243,5,18,13
466,6,20,14
370,4,19,15
125,3,18,15



==================== READABILITY CHECK, ACTUAL EXTERNAL FILES ====================
Readability:
read_ok
True    8508
Name: count, dtype: int64

Mode by batch:
mode     RGB
batch       
batch1  6005
batch2  2503

Dimension summary by batch:


width                                                          \
         count         mean         std    min     25%     50%     75%   
batch                                                                    
batch1  6005.0   769.412490  144.362172   89.0   646.0   803.0   853.0   
batch2  2503.0  1116.759089  124.933360  720.0  1024.0  1024.0  1260.0   

                height                                                     \
           max   count        mean        std    min    25%    50%    75%   
batch                                                                       
batch1  1171.0  6005.0  573.850291  69.063670   81.0  542.0  587.0  606.0   
batch2  1552.0  2503.0  822.629644  83.413563  576.0  768.0  768.0  910.0   

               
          max  
batch          
batch1  806.0  
batch2  970.0


==================== FINAL EXTERNAL ACTUAL-FILE USABLE MANIFEST ====================

==================== Full external actual-file usable ====================
Rows/images: 8500
Unique case_uid: 842
Missing labels: 0
Image exists: {True: 8500}

Image-level label distribution:
label
0    3097
1    5403
Name: count, dtype: int64
label
0    36.44
1    63.56
Name: proportion, dtype: float64

Case-level label distribution:
label
0    318
1    524
Name: count, dtype: int64
label
0    37.77
1    62.23
Name: proportion, dtype: float64

Images per case summary:
count    842.000000
mean      10.095012
std        4.664282
min        1.000000
25%        7.000000
50%        9.000000
75%       13.000000
max       31.000000
dtype: float64

Usable images by batch:
batch
batch1    6005
batch2    2495
Name: count, dtype: int64

Usable cases by batch:
batch
batch1    601
batch2    241
Name: case_uid, dtype: int64

Missing-label cases in actual-file raw manifest:


,batch,patient_name,case_uid
7148,batch2,200,batch2:200



Expected-paper-aligned check:
Raw actual images: 8508
Raw actual cases: 843
Usable images: 8500
Usable cases: 842
Duplicate image paths: 0
Cases with inconsistent labels: 0

==================== SAVED ====================
/content/drive/MyDrive/thyroid_external_validation_outputs/manifest_batch1_actual_file_based.csv
/content/drive/MyDrive/thyroid_external_validation_outputs/manifest_batch2_actual_file_based.csv
/content/drive/MyDrive/thyroid_external_validation_outputs/manifest_hou2024_external_actual_file_raw.csv
/content/drive/MyDrive/thyroid_external_validation_outputs/manifest_hou2024_external_actual_file_usable.csv
/content/drive/MyDrive/thyroid_external_validation_outputs/cell4_external_actual_readability.csv
/content/drive/MyDrive/thyroid_external_validation_outputs/cell4_batch1_csv_vs_actual_patient_counts.csv
/content/drive/MyDrive/thyroid_external_validation_outputs/cell4_external_actual_manifest_summary.json

==================== CELL 4 SUMMARY ====================
{
  "ba